In [8]:
import numpy as np
import pandas as pd
from pathlib import Path

from eval import eval_scGRN_SVGRN

root = Path(r"E:/st/SVGRN_subcellular")
result_path = root / "runs/stage2_subcellular/20260224_165408"
gt_scGRN_folder = root / "in_sim/g110_c2k_n01/cell_specific_GRN"
tf_file = root / "in_sim/g110_c2k_n01/TF_list.txt"
expr_file = root / "in_sim/g110_c2k_n01/expression_loc_cluster_wlayout.csv"
cellname_file = root / "in_sim/g110_c2k_n01/cellname_list_2000.npy"

cell_name_list = np.load(cellname_file, allow_pickle=True).astype(str).ravel().tolist()
all_data = pd.read_csv(expr_file, index_col=0)
gene_list = [str(g) for g in all_data.drop(columns=["x", "y", "ClusterID"], errors="ignore").columns]
TF_list = [line.strip() for line in tf_file.read_text(encoding="utf-8").splitlines() if line.strip()]

print(f"result_path: {result_path}")
print(f"gt_scGRN_folder: {gt_scGRN_folder}")
print(f"n_cells in list: {len(cell_name_list)}")
print(f"n_genes: {len(gene_list)}")
print(f"n_TFs: {len(TF_list)}")


result_path: E:\st\SVGRN_subcellular\runs\stage2_subcellular\20260224_165408
gt_scGRN_folder: E:\st\SVGRN_subcellular\in_sim\g110_c2k_n01\cell_specific_GRN
n_cells in list: 2000
n_genes: 110
n_TFs: 6


In [9]:
from eval import get_evaluation_matrix, get_truth_edges, evaluate, get_AUPRC_AUROC

Evaluate_Mask = get_evaluation_matrix(TF_list, gene_list)

TP_num_list, EP_rate_list, EPR_list = [], [], []
AUPRC_list, AUPRC_rate_list, AUROC_list = [], [], []
processed = 0

for cell_name in cell_name_list:
    csv_path = result_path / cell_name / "RN_150.csv"
    if not csv_path.exists():
        continue

    pre_GRN_i = pd.read_csv(csv_path).values.T
    truth_edges = get_truth_edges(str(gt_scGRN_folder / f"{cell_name}.csv"))

    TP_num, EP_rate, EPR = evaluate(pre_GRN_i, truth_edges, Evaluate_Mask=Evaluate_Mask, cutoff_factor=1)
    TP_num_list.append(TP_num)
    EP_rate_list.append(EP_rate)
    EPR_list.append(EPR)

    auprc, auprc_ratio, auroc = get_AUPRC_AUROC(pre_GRN_i, truth_edges, Evaluate_Mask=Evaluate_Mask)
    if auprc is not None:
        AUPRC_list.append(auprc)
        AUPRC_rate_list.append(auprc_ratio)
        AUROC_list.append(auroc)

    processed += 1
    if processed % 200 == 0:
        print(f"Processed {processed} cells")

if processed == 0:
    raise RuntimeError("No RN_150.csv found under result_path")

ave_TP_num = sum(TP_num_list) / len(TP_num_list)
ave_EP_rate = sum(EP_rate_list) / len(EP_rate_list)
ave_EPR = sum(EPR_list) / len(EPR_list)
ave_AUPRC = sum(AUPRC_list) / len(AUPRC_list)
ave_AUPRC_ratio = sum(AUPRC_rate_list) / len(AUPRC_rate_list)
ave_AUROC = sum(AUROC_list) / len(AUROC_list)

print("----- Stage2 Summary -----")
print(f"TP: {ave_TP_num}")
print(f"EP: {ave_EP_rate}")
print(f"EPR: {ave_EPR}")
print(f"AUPRC: {ave_AUPRC}")
print(f"AUPRC Ratio: {ave_AUPRC_ratio}")
print(f"AUROC: {ave_AUROC}")


Processed 200 cells
Processed 400 cells
Processed 600 cells
Processed 800 cells
Processed 1000 cells
Processed 1200 cells
Processed 1400 cells
Processed 1600 cells
Processed 1800 cells
Processed 2000 cells
----- Stage2 Summary -----
TP: 29.0655
EP: 0.22358076923076622
EPR: 1.1351023668639055
AUPRC: 0.22017744095928554
AUPRC Ratio: 1.1178239310240645
AUROC: 0.5339627612481853


In [5]:
from eval import get_evaluation_matrix, get_truth_edges, evaluate, get_AUPRC_AUROC

stage1_csv = root / "runs/stage1_subcellular/20260224_164748/RN_150.csv"
if not stage1_csv.exists():
    raise FileNotFoundError(stage1_csv)

# align to evaluator convention: row=sender, col=receiver
stage1_A = pd.read_csv(stage1_csv).values.T
Evaluate_Mask = get_evaluation_matrix(TF_list, gene_list)

TP_num_list, EP_rate_list, EPR_list = [], [], []
AUPRC_list, AUPRC_rate_list, AUROC_list = [], [], []

for i, cell_name in enumerate(cell_name_list):
    truth_edges = get_truth_edges(str(gt_scGRN_folder / f"{cell_name}.csv"))
    TP_num, EP_rate, EPR = evaluate(stage1_A, truth_edges, Evaluate_Mask=Evaluate_Mask, cutoff_factor=1)
    TP_num_list.append(TP_num)
    EP_rate_list.append(EP_rate)
    EPR_list.append(EPR)

    auprc, auprc_ratio, auroc = get_AUPRC_AUROC(stage1_A, truth_edges, Evaluate_Mask=Evaluate_Mask)
    if auprc is not None:
        AUPRC_list.append(auprc)
        AUPRC_rate_list.append(auprc_ratio)
        AUROC_list.append(auroc)

    if (i + 1) % 200 == 0:
        print(f"Processed {i + 1}/{len(cell_name_list)} cells")

print("----- Stage1 Summary -----")
print(f"TP: {sum(TP_num_list) / len(TP_num_list)}")
print(f"EP: {sum(EP_rate_list) / len(EP_rate_list)}")
print(f"EPR: {sum(EPR_list) / len(EPR_list)}")
if AUPRC_list:
    print(f"AUPRC: {sum(AUPRC_list) / len(AUPRC_list)}")
    print(f"AUPRC_ratio: {sum(AUPRC_rate_list) / len(AUPRC_rate_list)}")
    print(f"AUROC: {sum(AUROC_list) / len(AUROC_list)}")


Processed 200/2000 cells
Processed 400/2000 cells
Processed 600/2000 cells
Processed 800/2000 cells
Processed 1000/2000 cells
Processed 1200/2000 cells
Processed 1400/2000 cells
Processed 1600/2000 cells
Processed 1800/2000 cells
Processed 2000/2000 cells
----- Stage1 Summary -----
TP: 28.3395
EP: 0.2179961538461505
EPR: 1.106749704142013
AUPRC: 0.2168023623948639
AUPRC_ratio: 1.1006889167739207
AUROC: 0.5241398004354146
